# Using `api_tool()` Against an Existing OpenAPI Service

This notebook demonstrates a common production pattern: a team already has a documented REST API, and the agent needs to query several related endpoints without hand-writing one wrapper function per endpoint.

Before you run it, export your own `OPENAI_API_KEY` into the notebook environment. The walkthrough assumes the key is available, but it does not persist credentials into the notebook itself.

The walkthrough starts a small local FastAPI service, points `api_tool()` at that service's base URL, and then runs an Agentspan agent that can inspect service ownership, deployments, incidents, and runbooks through the discovered API operations.

## Why this pattern matters

The pain point is not making a single HTTP call. That is what `http_tool()` is for. The pain point is maintaining many wrappers around one API when the spec already describes the available operations.

This notebook keeps the demo small, but the mechanism is the same one you would use for internal service catalogs, support systems, billing APIs, or operational control planes.

In [ ]:
import os
import shlex
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen

from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
PYTHON = os.environ.get('PYTHON', 'python3')
API_PORT = 8010
API_LISTEN_HOST = '0.0.0.0'
SERVER_PORT = 6770
PRIVATE_IP = subprocess.check_output('hostname -I', shell=True, text=True).strip().split()[0]
API_BASE_URL = os.environ.get('DEMO_API_BASE_URL', f'http://{PRIVATE_IP}:{API_PORT}')
API_SPEC_URL = os.environ.get('DEMO_API_SPEC_URL', f'{API_BASE_URL}/openapi.json')
SERVER_ROOT = f'http://127.0.0.1:{SERVER_PORT}'
SERVER_API = f'{SERVER_ROOT}/api'
os.environ['AGENTSPAN_SERVER_URL'] = SERVER_API
os.environ['DEMO_API_BASE_URL'] = API_BASE_URL
os.environ['DEMO_API_SPEC_URL'] = API_SPEC_URL
os.environ.setdefault('AGENTSPAN_LLM_MODEL', 'openai/gpt-4o-mini')
if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY to your own key before running this notebook.')

def run(cmd: str, check: bool = True) -> None:
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, executable='/bin/bash', text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed: {cmd}')

def start_tmux(session_name: str, command: str) -> None:
    run(f'tmux kill-session -t {session_name}', check=False)
    run(f'tmux new-session -d -s {session_name} {shlex.quote(command)}')

def wait_http(url: str, timeout_seconds: int = 30) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with urlopen(url, timeout=2) as response:
                if response.status == 200:
                    return
        except Exception:
            time.sleep(1)
    raise RuntimeError(f'timed out waiting for {url}')

def execution_link(execution_id: str, label: str) -> HTML:
    return HTML(f'<a href="{SERVER_ROOT}/execution/{execution_id}" target="_blank">{label}</a>')

display(Markdown(f'- API base URL: `{API_BASE_URL}`'))
display(Markdown(f'- OpenAPI spec URL: `{API_SPEC_URL}`'))
display(Markdown(f'- Agentspan UI: `{SERVER_ROOT}`'))
display(Markdown(f'- Model: `{os.environ["AGENTSPAN_LLM_MODEL"]}`'))


## Start the demo service

The local FastAPI app acts like a small internal operations API. Because it exposes a normal OpenAPI document, `api_tool()` can discover the operations without any hand-written tool wrappers.

In [ ]:
start_tmux(
    'api-tool-service',
    f"bash -lc 'cd {ROOT} && {PYTHON} -m uvicorn service_catalog_api:app --host {API_LISTEN_HOST} --port {API_PORT}'",
)
wait_http(f'{API_BASE_URL}/openapi.json')
run(f'curl -s {API_BASE_URL}/healthz')


## Inspect the operations the service publishes

This is the key reason `api_tool()` exists. The API already describes the available operations. The agent runtime can discover them directly from the spec instead of making you re-declare each one as a separate tool.

In [ ]:
from api_tool_demo import list_operations

list_operations(API_SPEC_URL)


## Start the local Agentspan server

This notebook starts a dedicated local Agentspan server on port `6770`. The server manages execution state and renders the UI. The notebook talks to the API, while the browser can inspect the same run by `execution_id`.

In [ ]:
run('agentspan server stop', check=False)
start_tmux(
    'api-tool-server',
    f"bash -lc 'agentspan server start --port {SERVER_PORT}'",
)
wait_http(f'{SERVER_ROOT}/actuator/health', timeout_seconds=60)
run(f'curl -s {SERVER_ROOT}/actuator/health')


## Define an agent with `api_tool()`

The important part is visible here in the notebook: `api_tool()` points at the OpenAPI document, not a hand-written Python wrapper. This demo also narrows the tool surface to the three operations the assistant actually needs, which keeps the runtime menu small and the execution graph easier to read.

In [ ]:
from agentspan.agents import Agent, AgentRuntime, api_tool

service_api = api_tool(
    url=API_SPEC_URL,
    name='service_catalog_api',
    tool_names=['get_service_details', 'get_latest_deployment', 'get_service_runbook'],
    max_tools=3,
)

assistant = Agent(
    name='service_ops_assistant',
    model=os.environ['AGENTSPAN_LLM_MODEL'],
    instructions=(
        'You help on-call engineers answer questions about service ownership, '
        'production deployments, incidents, and runbooks. Use the API tools for facts. '
        'If a runbook URL is available, include it directly.'
    ),
    tools=[service_api],
)


## Run the end-to-end demo

This prompt requires several related operations. A useful `api_tool()` demo should force the runtime to discover more than one endpoint from the same API surface.

In [ ]:
with AgentRuntime() as runtime:
    result = runtime.run(
        assistant,
        'A checkout incident is open. Tell me who owns checkout-api, what the latest production deployment changed, and which runbook I should open first.',
    )

result.print_result()
execution_id = result.execution_id
display(execution_link(execution_id, 'Open this execution in the Agentspan UI'))


## Use `tool_names` when you only want part of the spec

One practical pattern is narrowing the API surface to the operations an agent actually needs. That keeps the tool menu smaller and makes the agent's choices more legible.

In [ ]:
focused_api = api_tool(
    url=API_SPEC_URL,
    name='service_catalog_focus',
    tool_names=['get_service_details', 'get_latest_deployment'],
    max_tools=2,
)

focused_assistant = Agent(
    name='service_release_assistant',
    model=os.environ['AGENTSPAN_LLM_MODEL'],
    instructions='Answer service ownership and release questions using the focused API tools.',
    tools=[focused_api],
)

with AgentRuntime() as runtime:
    focused_result = runtime.run(
        focused_assistant,
        'Give me the owner, latest production deployment status, and runbook URL for inventory-api.',
    )

focused_result.print_result()
display(execution_link(focused_result.execution_id, 'Open the focused execution in the UI'))


## Notes to carry back into production

- `api_tool()` is a fit when the API already has a usable OpenAPI, Swagger, or Postman description.
- `http_tool()` is still the better choice for a single stable endpoint.
- `@tool` is the right fit when the logic is not already exposed as an HTTP API or when you need local Python dependencies.
- `tool_names` and `max_tools` matter once an API grows beyond a few obvious operations.